# Auction Results — Bronze to Silver

This notebook transforms raw auction lot results from the **bronze** layer into a curated **silver** Delta table for downstream reporting and analytics.

It works at the **lot level**, meaning each row represents an individual lot listed in an auction. The notebook standardizes the raw fields, derives business-friendly attributes, and loads the results into `company_db.silver.auctionresults`.

## What this notebook is doing

This sheet provides a breakdown of each auction lot, including:
* whether the lot was **sold** or **unsold**
* whether the lot was a **solid** lot or a **mixed** lot
* how much the lot sold for
* the estimate range and midpoint estimate
* key product attributes such as country, region, variety, format, and item_year

## Main transformations

1. Reads raw lot-level results from `company_db.bronze.auctionresults`.
2. Removes raw operational columns that are not needed in silver.
3. Renames fields to cleaner analytics-friendly names.
4. Splits the auction date range into `start_date` and `end_date`.
5. Derives business fields such as:
   * `auction_category` = `Mixed` or `Solid`
   * `status` = `Sold` or `Unsold`
   * `midestimate` = average of low and high estimate
   * standardized `region` and `country`
6. Casts pricing and numeric fields to numeric types.
7. Creates the target Delta table in the silver layer.
8. Merges transformed records into silver so existing lots are updated and new lots are inserted.

## Source and target

* **Source:** `company_db.bronze.auctionresults`
* **Target:** `company_db.silver.auctionresults`
* **Storage location:** `abfss://auction@company.dfs.core.windows.net/silver/auctionresults`

## Data dictionary

| Column | Meaning |
| --- | --- |
| `auctionname` | Name of the auction event. |
| `auctiondate` | Original auction date range text from the source system. |
| `start_date` | Parsed auction start date. |
| `end_date` | Parsed auction end date. |
| `item_id` | Source item identifier for the lot. |
| `lot_name` | Display name or title of the lot. |
| `lot_number` | Lot number used in the auction catalog. |
| `format` | Packaging format for the lot. |
| `country` | Country associated with the lot. |
| `region` | Reporting region used for analytics. For France, major regions are preserved and all others roll up to `France(Others)`. |
| `sub_region` | Sub-region from the source attributes. |
| `locality` | Local area detail where available. |
| `variety` | Variety variety associated with the lot. |
| `type` | Item type or category attribute. |
| `item_year` | Item year. Missing values are filled with `N/A`. |
| `category` | Original source category value. |
| `auction_category` | Business classification of the lot as `Mixed` or `Solid`. |
| `status` | Sale outcome for the lot. `Sold` means a realized price exists; `Unsold` means the source used `--` and the notebook converts the price to `0`. |
| `startingbid` | Opening bid amount for the lot. |
| `lowestimate` | Low estimate for the lot. |
| `highestimate` | High estimate for the lot. |
| `midestimate` | Midpoint of `lowestimate` and `highestimate`. |
| `hammerprice` | Hammer price recorded for the lot. |
| `pricerealized` | Final realized sale value used to analyze how much the lot sold for. Unsold lots are stored as `0`. |
| `buyerspremium` | Buyer premium amount associated with the lot. |
| `commissionamount` | Commission amount associated with the lot. |
| `totalsettlement` | Settlement total recorded for the consignor side. |
| `favoritescount` | Number of users who favorited the lot. |
| `path` | Source file or ingestion path used for traceability. |
| `ingestion_timestamp` | Timestamp when the raw record was ingested. |
| `created_at` | Timestamp when the record was first inserted into silver. |
| `updated_at` | Timestamp when the record was last updated in silver. |

## Business interpretation

* A **sold lot** has a realized value and is labeled `Sold`.
* An **unsold lot** is labeled `Unsold` and has price fields normalized from `--` to `0`.
* A **mixed lot** contains multiple items grouped together.
* A **solid lot** is treated as a single-item or non-mixed lot.

This silver table is intended to support reporting on auction performance, sell-through, pricing, and lot composition at the individual lot level.

In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables" 

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# Import PySpark SQL functions used for column transformations, casting, parsing, and conditional logic.
from pyspark.sql import functions as F
from pyspark.sql.functions import split

In [ ]:
# Read the raw auction results table from the bronze layer.
# This DataFrame is the source for all lot-level cleaning and business transformations.
auctionresults = spark.table(f'{catalog}.bronze.auctionresults').filter(F.col('batch_id')==batch_id)

In [ ]:
from src.transformations.auction_results import transform_auction_results

In [ ]:
# Apply transformations
auctionresults = transform_auction_results(auctionresults)

In [ ]:
auction_sales_report = spark.table(
    f'{catalog}.silver.auction_sales_report').filter(F.col('batch_id')==batch_id)
auction_sales_report = auction_sales_report.select('start_date','auction_id')
auctionresults = auctionresults.join(auction_sales_report,'start_date','left')

In [ ]:
item_desc = spark.table(f'{catalog}.bronze.item_desc').filter(F.col('batch_id')==batch_id).select('lot_number','item_name','quantity')

In [ ]:
auctionresults = auctionresults.join(item_desc, on='lot_number', how='left')

In [ ]:
# Create or replace the silver Delta table that stores curated lot-level auction results.
# This schema supports reporting on sold vs unsold lots, solid vs mixed lots, and realized pricing.

spark.sql(
f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.auctionresults (
    auction_id integer,
    auction_name STRING,
    auctiondate STRING,
    start_date DATE,
    end_date DATE,
    lot_id DOUBLE,
    lot_name STRING,
    item_name STRING,
    lot_number DOUBLE,
    quantity INT,
    link STRING,
    provenance STRING,
    exportedimgs STRING,
    `consignor_id` DOUBLE,
    consignor_name STRING,
    consignor_email STRING,
    `consignment_id` STRING,
    totalsettlement DOUBLE,
    format STRING,
    `attribute:country` STRING,
    `attribute:region` STRING,
    region STRING,
    country STRING,
    sub_region STRING,
    locality STRING,
    variety STRING,
    type STRING,
    item_year STRING,
    category STRING,
    lotdescription STRING,
    condition STRING,
    startingbid DOUBLE,
    lowestimate DOUBLE,
    highestimate DOUBLE,
    midestimate DOUBLE,
    hammerprice DOUBLE,
    commissionamount DOUBLE,
    buyerspremium DOUBLE,
    pricerealized DOUBLE,
    favoritescount DOUBLE,
    customfields STRING,
    lot_category STRING,
    status STRING,
    path STRING,
    ingestion_timestamp TIMESTAMP,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    batch_id STRING
    
    
)
USING DELTA
LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/silver/auctionresults';""")

In [ ]:
# Register the transformed DataFrame as a temporary view so it can be used by the SQL MERGE statement.
auctionresults.createOrReplaceTempView('auctionresults_view')

In [ ]:
# Upsert the transformed lot-level records into the silver auction results table.
# Existing rows are updated based on item_id + lot_number + start_date, and new rows are inserted.

spark.sql(
f"""
MERGE INTO {catalog}.silver.auctionresults as target
USING auctionresults_view as source
ON target.lot_id = source.lot_id
AND
target.lot_number = source.lot_number
And target.auction_id = source.auction_id

WHEN MATCHED 
AND target.batch_id >= source.batch_id 
THEN
UPDATE SET
target.auction_name = source.auction_name,
target.auction_id = source.auction_id,
target.auctiondate = source.auctiondate,
target.start_date = source.start_date,
target.end_date = source.end_date,
target.lot_id = source.lot_id,
target.lot_name = source.lot_name,
target.item_name = source.item_name,
target.lot_number = source.lot_number,
target.quantity = source.quantity,
target.link = source.link,
target.provenance = source.provenance,
target.exportedimgs = source.exportedimgs,
target.`consignor_id` = source.`consignor_id`,
target.consignor_name = source.consignor_name,
target.consignor_email = source.consignor_email,
target.`consignment_id` = source.`consignment_id`,
target.totalsettlement = source.totalsettlement,
target.format = source.format,
target.`attribute:country` = source.`attribute:country`,
target.`attribute:region` = source.`attribute:region`,
target.region = source.region,
target.country = source.country,
target.sub_region = source.sub_region,
target.locality = source.locality,
target.variety = source.variety,
target.type = source.type,
target.item_year = source.item_year,
target.category = source.category,
target.lotdescription = source.lotdescription,
target.condition = source.condition,
target.startingbid = source.startingbid,
target.lowestimate = source.lowestimate,
target.highestimate = source.highestimate,
target.midestimate = source.midestimate,
target.hammerprice = source.hammerprice,
target.commissionamount = source.commissionamount,
target.buyerspremium = source.buyerspremium,
target.pricerealized = source.pricerealized,
target.favoritescount = source.favoritescount,
target.customfields = source.customfields,
target.lot_category = source.lot_category,
target.status = source.status,
target.path = source.path,
target.ingestion_timestamp = source.ingestion_timestamp,
target.updated_at = current_timestamp(),
target.batch_id = source.batch_id
WHEN NOT MATCHED THEN
   INSERT (auction_name,auction_id, auctiondate,start_date,end_date, lot_id, lot_name, item_name, lot_number, quantity, link, provenance, exportedimgs, `consignor_id`, consignor_name, consignor_email, `consignment_id`, totalsettlement, format, `attribute:country`, `attribute:region`, region, country, sub_region, locality, variety, type, item_year, category, lotdescription, condition, startingbid, lowestimate, highestimate, midestimate, hammerprice, commissionamount, buyerspremium, pricerealized, favoritescount, customfields, lot_category, status, path, ingestion_timestamp, updated_at, created_at,batch_id)
   VALUES (source.auction_name, source.auction_id, source.auctiondate,source.start_date,source.end_date, source.lot_id, source.lot_name, source.item_name, source.lot_number, source.quantity, source.link, source.provenance, source.exportedimgs, source.`consignor_id`, source.consignor_name, source.consignor_email, source.`consignment_id`, source.totalsettlement, source.format, source.`attribute:country`, source.`attribute:region`, source.region, source.country, source.sub_region, source.locality, source.variety, source.type, source.item_year, source.category, source.lotdescription, source.condition, source.startingbid, source.lowestimate, source.highestimate, source.midestimate, source.hammerprice, source.commissionamount, source.buyerspremium, source.pricerealized, source.favoritescount, source.customfields, source.lot_category, source.status, source.path, source.ingestion_timestamp, current_timestamp(), current_timestamp(), source.batch_id)""")

## Analyst notes

Use this table when you need lot-level auction reporting such as:
* sell-through analysis by auction
* sold vs unsold counts
* mixed vs solid lot mix
* realized price analysis
* estimate vs realized performance
* regional or year-on-year performance trends

Because the notebook stores one row per lot, it is well suited for dashboards and downstream silver-to-gold aggregations.

## Analyst notes

Use this table when you need lot-level auction reporting such as:
* sell-through analysis by auction
* sold vs unsold counts
* mixed vs solid lot mix
* realized price analysis
* estimate vs realized performance
* regional or year-on-year performance trends

Because the notebook stores one row per lot, it is well suited for dashboards and downstream silver-to-gold aggregations.